# 🧠 Multi-Phase Exploratory Data Analysis: Human Decision Fatigue & Performance Degradation
> **Course**: Data Visualization  
> **Dataset**: `human_decision_fatigue_dataset.csv` (25,000 entries)
> **Approach**: Sequential Hypothesis Testing & High-Dimensional Interactive Visualization with **Plotly**.

This notebook is organized into three distinct analytical phases:
1. **Phase 1: Testing Bivariate Preliminary Hypotheses (Plots A to E)** - Establishing baseline relations and core inputs sequentially from cause to effect.
2. **Phase 2: Premium High-Dimensional Interactions (Plots F to H)** - Exploring complex 3-variable and 4-variable interactions where cognitive systems hit tipping points or collapses.
3. **Phase 3: Managerial Utility & Policy Visualizations (Plots I to L)** - Testing specific corporate policy limits and interventions from a management/employee wellness perspective.


In [1]:
import sys
import os
import subprocess

# Self-checking robust imports for clean notebook kernels
try:
    import pandas as pd
    import numpy as np
    import plotly.express as px
    import plotly.graph_objects as go
    import plotly.io as pio
except ImportError as e:
    print(f"⚠️ [IMPORT WARNING]: Missing dependency in current kernel: {e}")
    print("Attempting to auto-install required visualization libraries under current environment...")
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "pandas", "numpy", "plotly", "statsmodels", "scikit-learn"])
        import pandas as pd
        import numpy as np
        import plotly.express as px
        import plotly.graph_objects as go
        import plotly.io as pio
        print("✅ Libraries installed and imported successfully!")
    except Exception as install_err:
        raise ImportError(
            f"Failed to auto-install missing packages: {install_err}. "
            "Please select the project virtual environment (.venv) at the top-right corner of your Jupyter interface."
        )

# Self-healing patch to resolve recursive show() monkey-patches from previous Jupyter sessions
try:
    go.Figure.show = pio.show
except Exception:
    pass

# Configure premium, Tufte-maximizing Cool Slate-Blue light theme globally
pio.templates["premium_tufte"] = go.layout.Template(
    layout=go.Layout(
        paper_bgcolor='#F4F6FA',  # Warm off-white / Cool slate-blue canvas
        plot_bgcolor='#F4F6FA',
        font=dict(color='#2B3A42', family="Inter, Roboto, Helvetica, Arial, sans-serif"),
        title=dict(font=dict(size=16, color='#2B3A42', weight='bold')),
        legend=dict(
            bgcolor='rgba(0,0,0,0)',      # Floating transparent background
            bordercolor='rgba(0,0,0,0)',  # Strip borders to maximize data-ink
            borderwidth=0
        ),
        xaxis=dict(
            showline=False,               # Remove spine borders
            gridcolor='#E1E6EB',          # Desaturated soft gridlines
            zeroline=False,
            tickfont=dict(color='#4F5B66')
        ),
        yaxis=dict(
            showline=False,               # Remove spine borders
            gridcolor='#E1E6EB',          # Desaturated soft gridlines
            zeroline=False,
            tickfont=dict(color='#4F5B66')
        ),
        scene=dict(
            xaxis=dict(backgroundcolor='#F4F6FA', gridcolor='#E1E6EB', showbackground=True),
            yaxis=dict(backgroundcolor='#F4F6FA', gridcolor='#E1E6EB', showbackground=True),
            zaxis=dict(backgroundcolor='#F4F6FA', gridcolor='#E1E6EB', showbackground=True)
        )
    )
)
pio.templates.default = "plotly_white+premium_tufte"

print("🚀 Environment loaded with Tufte-Maximizing Cool Slate-Blue light style!")


🚀 Environment loaded with Tufte-Maximizing Cool Slate-Blue light style!


In [2]:
import os
# Programmatically resolve dataset path to run seamlessly from both the workspace root or the src directory
csv_path = 'data/human_decision_fatigue_dataset.csv'
if not os.path.exists(csv_path):
    csv_path = '../data/human_decision_fatigue_dataset.csv'

df = pd.read_csv(csv_path)

# Configure categorical orders for logical sorting
df['Time_of_Day'] = pd.Categorical(df['Time_of_Day'], categories=['Morning', 'Afternoon', 'Evening', 'Night'], ordered=True)
df['Fatigue_Level'] = pd.Categorical(df['Fatigue_Level'], categories=['Low', 'Moderate', 'High'], ordered=True)
df['System_Recommendation'] = pd.Categorical(df['System_Recommendation'], categories=['Continue', 'Slow Down', 'Take Break'], ordered=True)

# Create sleep and stress groups for box plots and policy limits
def sleep_bin(h):
    if h < 6: return 'Short Sleep (<6h)'
    elif h <= 8: return 'Standard Sleep (6-8h)'
    else: return 'Optimal Sleep (>8h)'

df['Sleep_Group'] = df['Sleep_Hours_Last_Night'].apply(sleep_bin)
df['Sleep_Group'] = pd.Categorical(df['Sleep_Group'], categories=['Short Sleep (<6h)', 'Standard Sleep (6-8h)', 'Optimal Sleep (>8h)'], ordered=True)

# Create a global down-sampled dataframe for heavy bubble/scatter plots to ensure fast performance
sample_df = df.sample(1500, random_state=42)

# Define global color palette mapping for System Recommendations
PALETTE_RECOMMENDATION = {"Continue": "#2ec4b6", "Slow Down": "#ff9f1c", "Take Break": "#e71d36"}

df.head()


,Hours_Awake,Decisions_Made,Task_Switches,Avg_Decision_Time_sec,Sleep_Hours_Last_Night,Time_of_Day,Caffeine_Intake_Cups,Stress_Level_1_10,Error_Rate,Cognitive_Load_Score,Decision_Fatigue_Score,Fatigue_Level,System_Recommendation,Sleep_Group
0,7,28,7,2.30,5.8,Evening,0,2.4,0.000,2.6,15.6,Low,Continue,Short Sleep (<6h)
1,15,77,22,3.65,4.5,Afternoon,3,1.9,0.143,4.5,97.3,High,Take Break,Short Sleep (<6h)
2,11,57,23,3.67,6.8,Night,2,2.1,0.000,4.1,55.4,Moderate,Slow Down,Standard Sleep (6-8h)
3,8,39,10,2.39,5.3,Afternoon,1,1.0,0.000,2.3,29.7,Low,Continue,Short Sleep (<6h)
4,7,46,16,3.05,8.2,Night,1,2.8,0.000,3.9,19.1,Low,Continue,Optimal Sleep (>8h)


# 📊 Phase 1: Testing Bivariate Preliminary Hypotheses (Roadmap A-E)
In this phase, we analyze the baseline relationships proposed in our initial testing roadmap to trace cause-and-effect sequentially from workload inputs to system outputs.


### 📈 Plot A: The Workload Threshold (Decisions vs. Task Switches)
* **Hypothesis:** Bivariate density thresholds of workload inputs (`Decisions_Made` and `Task_Switches`) correspond directly with high baseline `Cognitive_Load_Score`.
* **Visualization:** Interactive density heatmap grouping decisions and task switches, filled with average cognitive load.


In [3]:
pivot_workload = df.groupby([
    pd.cut(df['Decisions_Made'], bins=15),
    pd.cut(df['Task_Switches'], bins=15)
], observed=False)['Cognitive_Load_Score'].mean().unstack().fillna(0)

pivot_workload.index = [f"{int(i.left)}-{int(i.right)}" for i in pivot_workload.index]
pivot_workload.columns = [f"{int(c.left)}-{int(c.right)}" for c in pivot_workload.columns]

fig_a = px.imshow(
    pivot_workload,
    labels=dict(x="Number of Task Switches", y="Decisions Made", color="Avg Cognitive Load"),
    title="<b>Plot A: Bivariate Workload Grid vs. Average Cognitive Load</b>",
    color_continuous_scale="Viridis"
)
fig_a.update_layout(title_font_size=18, margin=dict(l=40, r=40, t=60, b=40))
fig_a.show()


### 🛌 Plot B: Sleep Deprivation vs. Stress Escalation (Grouped Comparison)
* **Hypothesis:** Operational stress levels (`Stress_Level_1_10`) escalate when sleep quality is deficient (<6h).
* **Visualization:** Grouped box plots displaying the distribution of stress levels across categorized sleep groups.


In [4]:
fig_b = px.box(
    df,
    x='Sleep_Group',
    y='Stress_Level_1_10',
    color='Sleep_Group',
    color_discrete_sequence=['#e71d36', '#ff9f1c', '#2ec4b6'],
    labels={
        'Sleep_Group': 'Sleep Duration Class',
        'Stress_Level_1_10': 'Stress Level (1-10)'
    },
    title="<b>Plot B: Stress Level Distribution by Sleep Quality Group</b>"
)
fig_b.update_layout(title_font_size=18, margin=dict(l=40, r=40, t=60, b=40))
fig_b.show()


### 🕰️ Plot C: The Fatigue-Performance Curve (Longitudinal Trend)
* **Hypothesis:** Fatigue and performance errors exhibit a non-linear accumulation wall as continuous time awake increases.
* **Visualization:** Custom dual-axis line chart mapping average `Decision_Fatigue_Score` and percentage `Error_Rate` across `Hours_Awake`.


In [5]:
grouped_awake = df.groupby('Hours_Awake')[['Error_Rate', 'Decision_Fatigue_Score']].mean().reset_index()

fig_c = go.Figure()

# Add line for Decision Fatigue Score
fig_c.add_trace(
    go.Scatter(
        x=grouped_awake['Hours_Awake'],
        y=grouped_awake['Decision_Fatigue_Score'],
        name='Decision Fatigue Score (Left)',
        line=dict(color='#ff9f1c', width=4),
        mode='lines+markers',
        marker=dict(size=8, color='#ff9f1c', line=dict(width=0))
    )
)

# Add line for Error Rate (on secondary Y-axis)
fig_c.add_trace(
    go.Scatter(
        x=grouped_awake['Hours_Awake'],
        y=grouped_awake['Error_Rate'] * 100,
        name='Error Rate % (Right)',
        line=dict(color='#e71d36', width=4, dash='dash'),
        mode='lines+markers',
        marker=dict(size=8, color='#e71d36', line=dict(width=0)),
        yaxis='y2'
    )
)

fig_c.update_layout(
    title='<b>Plot C: The Cognitive Wall: Hours Awake vs. Fatigue & Error Rate</b>',
    title_font_size=18,
    xaxis=dict(title='Hours Awake', dtick=1),
    yaxis=dict(title='Decision Fatigue Score (0-100)', title_font=dict(color='#ff9f1c'), tickfont=dict(color='#ff9f1c')),
    yaxis2=dict(
        title='Error Rate (%)',
        title_font=dict(color='#e71d36'),
        tickfont=dict(color='#e71d36'),
        overlaying='y',
        side='right'
    ),
    legend=dict(x=0.05, y=0.95),
    margin=dict(l=40, r=40, t=60, b=40),
    shapes=[
        dict(type="rect", xref="x", yref="paper", x0=7, x1=10, y0=0, y1=1, fillcolor="rgba(46, 196, 182, 0.15)", layer="below", line_width=0),
        dict(type="rect", xref="x", yref="paper", x0=10, x1=13, y0=0, y1=1, fillcolor="rgba(255, 159, 28, 0.15)", layer="below", line_width=0),
        dict(type="rect", xref="x", yref="paper", x0=13, x1=17, y0=0, y1=1, fillcolor="rgba(231, 29, 54, 0.25)", layer="below", line_width=0)
    ]
)

fig_c.add_annotation(x=8.5, y=50, text="<b>SAFETY ZONE</b>", showarrow=False, font=dict(color="#2ec4b6", size=10))
fig_c.add_annotation(x=11.5, y=65, text="<b>WARNING ZONE</b>", showarrow=False, font=dict(color="#ff9f1c", size=10))
fig_c.add_annotation(x=15, y=85, text="<b>DANGER ZONE</b>", showarrow=False, font=dict(color="#e71d36", size=10))

fig_c.show()


### ☕ Plot D: Caffeine Boost vs. Fatigue Crash (Clustered Heatmap)
* **Hypothesis:** Caffeine intake suppresses perceived fatigue over progressive time awake intervals.
* **Visualization:** Pivot heatmap mapping `Caffeine_Intake_Cups` vs. `Hours_Awake`, filled with average `Decision_Fatigue_Score`.


In [6]:
pivot_caffeine = df.groupby(['Hours_Awake', 'Caffeine_Intake_Cups'], observed=False)['Decision_Fatigue_Score'].mean().unstack().fillna(0)

fig_d = px.imshow(
    pivot_caffeine,
    labels=dict(x="Caffeine Intake (Cups)", y="Hours Awake", color="Avg Fatigue Score"),
    title="<b>Plot D: Caffeine Intake & Hours Awake vs. Perceived Decision Fatigue</b>",
    color_continuous_scale="Portland"
)
fig_d.update_layout(title_font_size=18, margin=dict(l=40, r=40, t=60, b=40))
fig_d.show()


### 🛑 Plot E: Recommendation Boundary (2D Bubble Space)
* **Hypothesis:** The automated recommendation engine is not a simple linear trigger, but divides action boundaries across multi-dimensional features.
* **Visualization:** 2D Bubble chart mapping `Cognitive_Load_Score` (X) vs. `Decision_Fatigue_Score` (Y), where bubble size encodes `Error_Rate` and color represents the `System_Recommendation`.


In [7]:
fig_e = px.scatter(
    sample_df,
    x='Cognitive_Load_Score',
    y='Decision_Fatigue_Score',
    color='System_Recommendation',
    size='Error_Rate',
    color_discrete_map=PALETTE_RECOMMENDATION,
    size_max=12,
    labels={
        'Cognitive_Load_Score': 'Cognitive Load Score',
        'Decision_Fatigue_Score': 'Decision Fatigue Score',
        'Error_Rate': 'Error Rate',
        'System_Recommendation': 'System Action'
    },
    title='<b>Plot E: Recommendation Decision Space & Action Boundaries (2D Bubble Chart)</b>',
    opacity=0.75
)
fig_e.update_traces(marker=dict(line=dict(width=0)))
fig_e.update_layout(title_font_size=18, margin=dict(l=40, r=40, t=60, b=40))
fig_e.show()


# 🌀 Phase 2: Premium High-Dimensional Interactions (Roadmap F-H)
Having validated the baseline relations, we now analyze three complex 4-variable interactions where multiple environmental, cognitive, and stress factors compound to trigger system crashes or functional failures.


### 🌪️ Plot F: The "Perfect Storm" Catastrophe Zone (Faceted 2D Scatter)
* **The Non-Obvious:** Having low sleep (< 6h) combined with prolonged wakefulness (>= 12h) and high stress (>= 7) forms a **catastrophic performance zone** with a 15.3% error rate. Good sleep isolates this error rate down to 7.2%, proving sleep serves as a buffer against stress and fatigue failures.
* **Visualization:** Faceted 2D Bubble Plot across `Sleep_Group` classes, mapping `Hours_Awake` (X) vs. `Error_Rate` (Y), colored by `Stress_Level_1_10` and sized by `Cognitive_Load_Score`.


In [8]:
fig_f = px.scatter(
    sample_df,
    x='Hours_Awake',
    y='Error_Rate',
    facet_col='Sleep_Group',
    color='Stress_Level_1_10',
    size='Cognitive_Load_Score',
    color_continuous_scale='Hot',
    size_max=12,
    labels={
        'Hours_Awake': 'Hours Awake',
        'Error_Rate': 'Error Rate',
        'Sleep_Group': 'Sleep Duration Class',
        'Stress_Level_1_10': 'Stress level (1-10)',
        'Cognitive_Load_Score': 'Cognitive Load'
    },
    title='<b>Plot F: The "Perfect Storm" Catastrophe Zone: Sleep Buffering Stress & Fatigue</b>',
    opacity=0.8
)
fig_f.update_traces(marker=dict(line=dict(width=0)))
fig_f.update_layout(title_font_size=18, yaxis=dict(tickformat='.0%'), margin=dict(l=40, r=40, t=60, b=40))
fig_f.show()


### 📉 Plot G: Multitasking Efficiency Decay (2D Bubble Decay)
* **The Non-Obvious:** Making decisions under low task-switching has a flat speed cost. However, high multitasking frequency (>20 switches) causes decision speeds to stretch sluggishly, while cognitive load spikes exponentially.
* **Visualization:** 2D Bubble Chart mapping `Task_Switches` (X) vs. `Avg_Decision_Time_sec` (Y), colored by `Cognitive_Load_Score` and sized by `Decisions_Made`.


In [9]:
fig_g = px.scatter(
    sample_df,
    x='Task_Switches',
    y='Avg_Decision_Time_sec',
    color='Cognitive_Load_Score',
    size='Decisions_Made',
    color_continuous_scale='Bluered',
    size_max=12,
    labels={
        'Decisions_Made': 'Decisions Made',
        'Avg_Decision_Time_sec': 'Average Decision Time (s)',
        'Task_Switches': 'Task Switches',
        'Cognitive_Load_Score': 'Cognitive Load'
    },
    title='<b>Plot G: Multitasking Efficiency Decay: Speed sluggishness vs. Switching Friction</b>',
    opacity=0.8
)
fig_g.update_traces(marker=dict(line=dict(width=0)))
fig_g.update_layout(title_font_size=18, margin=dict(l=40, r=40, t=60, b=40))
fig_g.show()


### 🛑 Plot H: Speed-Accuracy Trade-off (SAT) Collapse (Faceted SAT Plots)
* **The Non-Obvious:** Under low and moderate fatigue, users successfully compensate for high stress by slowing down, maintaining a 0% error rate. Under high fatigue, this trade-off collapses completely: users work at maximum sluggishness and yet experience massive error rates (10%+).
* **Visualization:** Interactive faceted scatter plot mapping `Avg_Decision_Time_sec` (X) vs. `Error_Rate` (Y), faceted by `Fatigue_Level` class, and colored by `Stress_Level_1_10`.


In [10]:
fig_h = px.scatter(
    sample_df,
    x='Avg_Decision_Time_sec',
    y='Error_Rate',
    facet_col='Fatigue_Level',
    color='Stress_Level_1_10',
    color_continuous_scale='Viridis',
    labels={
        'Avg_Decision_Time_sec': 'Avg Decision Time (s)',
        'Error_Rate': 'Error Rate',
        'Fatigue_Level': 'Fatigue Level',
        'Stress_Level_1_10': 'Stress Level'
    },
    title='<b>Plot H: Speed-Accuracy Trade-off Collapse: Active Compensation vs. Total Collapse</b>',
    opacity=0.8
)
fig_h.update_traces(marker=dict(line=dict(width=0)))
fig_h.update_layout(yaxis=dict(tickformat='.0%'), margin=dict(l=40, r=40, t=60, b=40))
fig_h.show()


# 💼 Phase 3: Managerial Utility & Policy Visualizations (Roadmap I-L)
In this final phase, we test specific actionable workplace policies and interventions from a corporate operations and wellness management perspective.


### 🔄 Plot I: The "Context-Switching Cap" Policy (Multitasking vs. Speed)
* **Managerial Point of View:** Employees are not exhausted by the total quantity of decisions, but by the mental friction of multitasking.
* **Visualization:** Scatter plot of `Task_Switches` (X) vs. `Cognitive_Load_Score` (Y), with OLS regression line, and colored by `Avg_Decision_Time_sec` to show the joint decay of mental state and productivity.


In [11]:
fig_i = px.scatter(
    sample_df,
    x='Task_Switches',
    y='Cognitive_Load_Score',
    color='Avg_Decision_Time_sec',
    color_continuous_scale='Turbo',
    trendline='ols',
    trendline_color_override='#e71d36', # Premium Red high-contrast trendline for light canvas
    labels={
        'Task_Switches': 'Task Switches',
        'Cognitive_Load_Score': 'Cognitive Load Score',
        'Avg_Decision_Time_sec': 'Avg Decision Time (s)'
    },
    title='<b>Plot I: The Context-Switching Cap Policy: Multitasking vs. Mental Load & Speed</b>',
    opacity=0.8
)
fig_i.update_traces(selector=dict(mode='markers'), marker=dict(line=dict(width=0)))
fig_i.update_layout(title_font_size=18, margin=dict(l=40, r=40, t=60, b=40))
fig_i.show()


### 🛌 Plot J: The "Sleep-Aware Stress Mitigation" Policy
* **Managerial Point of View:** Employees starting with a sleep deficit react aggressively to decision workload. Sleep duration acts as a critical biological buffer.
* **Visualization:** Line chart mapping binned workload (`Decisions_Made`) vs. average `Stress_Level_1_10`, colored by `Sleep_Group`.


In [12]:
df_policy = df.copy()
df_policy['Workload_Bin'] = pd.cut(df_policy['Decisions_Made'], bins=6)
# Convert workload bins to clean text ranges
df_policy['Workload_Range'] = df_policy['Workload_Bin'].apply(lambda x: f"{int(x.left)}-{int(x.right)}")

grouped_policy = df_policy.groupby(['Sleep_Group', 'Workload_Range'], observed=False)['Stress_Level_1_10'].mean().reset_index()

# Ensure ranges sort logically
grouped_policy['Workload_Range'] = pd.Categorical(
    grouped_policy['Workload_Range'],
    categories=sorted(grouped_policy['Workload_Range'].unique(), key=lambda x: int(x.split('-')[0])),
    ordered=True
)

fig_j = px.line(
    grouped_policy.sort_values('Workload_Range'),
    x='Workload_Range',
    y='Stress_Level_1_10',
    color='Sleep_Group',
    color_discrete_sequence=['#e71d36', '#ff9f1c', '#2ec4b6'],
    markers=True,
    labels={
        'Workload_Range': 'Decision Workload (Decisions Made)',
        'Stress_Level_1_10': 'Avg Stress Level (1-10)',
        'Sleep_Group': 'Sleep Class'
    },
    title='<b>Plot J: The Sleep-Aware Mitigation Policy: Decision Stress Slopes by Sleep Class</b>'
)
fig_j.update_layout(title_font_size=18, margin=dict(l=40, r=40, t=60, b=40))
fig_j.show()


### 🕰️ Plot K: The "Overtime Fatigue Wall" Cap
* **Managerial Point of View:** Working beyond 10 continuous hours triggers an exponential safety liability. Shifts longer than 10 hours should be prohibited.
* **Visualization:** Bar chart showing average `Error_Rate` for each `Hours_Awake`, with a red dashed line at a target "Corporate Quality Tolerance" (e.g. 2% error rate).


In [13]:
hourly_errors = df.groupby('Hours_Awake')['Error_Rate'].mean().reset_index()

fig_k = px.bar(
    hourly_errors,
    x='Hours_Awake',
    y='Error_Rate',
    color='Error_Rate',
    color_continuous_scale='OrRd',
    labels={
        'Hours_Awake': 'Continuous Hours Awake',
        'Error_Rate': 'Average Error Rate'
    },
    title='<b>Plot K: The Overtime Fatigue Wall Cap: Hourly Error Rate vs. Corporate 2% Quality Tolerance Limit</b>'
)

# Add horizontal tolerance limit line
fig_k.add_shape(
    type="line",
    x0=6.5, x1=17.5,
    y0=0.02, y1=0.02, # 2% corporate tolerance threshold
    line=dict(color="#e71d36", width=3, dash="dash")
)

fig_k.add_annotation(
    x=10, y=0.025,
    text="⚠️ <b>Corporate Quality Tolerance Limit (2% Max Errors)</b>",
    showarrow=False,
    font=dict(color="#e71d36", size=11)
)

fig_k.update_layout(
    title_font_size=18,
    yaxis=dict(tickformat='.1%'),
    margin=dict(l=40, r=40, t=60, b=40)
)
fig_k.show()


### ☕ Plot L: The "False Alert" Stimulant Jitter Phenotype
* **Managerial Point of View:** Sleep-deprived employees consume heavy caffeine to speed up physical responses, but it triggers high stress and mistake rates. They work fast, but make errors at high speed.
* **Visualization:** Dual-facet line chart for sleep-deprived employees (`Sleep_Hours_Last_Night` < 6h) comparing `Avg_Decision_Time_sec` (speed) vs. `Error_Rate` (quality) across `Caffeine_Intake_Cups`.


In [14]:
sleep_deprived = df[df['Sleep_Hours_Last_Night'] < 6].copy()
caffeine_stats = sleep_deprived.groupby('Caffeine_Intake_Cups')[['Avg_Decision_Time_sec', 'Error_Rate']].mean().reset_index()

fig_l = go.Figure()

# Add line for Decision Time (Speed)
fig_l.add_trace(
    go.Scatter(
        x=caffeine_stats['Caffeine_Intake_Cups'],
        y=caffeine_stats['Avg_Decision_Time_sec'],
        name='Avg Decision Time (Seconds) - Productivity',
        line=dict(color='#2ec4b6', width=4),
        mode='lines+markers',
        marker=dict(size=9, color='#2ec4b6', line=dict(width=0))
    )
)

# Add line for Error Rate (Quality)
fig_l.add_trace(
    go.Scatter(
        x=caffeine_stats['Caffeine_Intake_Cups'],
        y=caffeine_stats['Error_Rate'] * 100,
        name='Error Rate (%) - Mistakes',
        line=dict(color='#e71d36', width=4, dash='dash'),
        mode='lines+markers',
        marker=dict(size=9, color='#e71d36', line=dict(width=0)),
        yaxis='y2'
    )
)

fig_l.update_layout(
    title='<b>Plot L: The False Alert Stimulant Jitter: Decision Speed Boost vs. Mistake Surge</b>',
    title_font_size=18,
    xaxis=dict(title='Caffeine Consumed (Cups during Shift)', dtick=1),
    yaxis=dict(title='Avg Decision Time (s) - Lower is Faster', title_font=dict(color='#2ec4b6'), tickfont=dict(color='#2ec4b6')),
    yaxis2=dict(
        title='Error Rate (%) - Lower is Better',
        title_font=dict(color='#e71d36'),
        tickfont=dict(color='#e71d36'),
        overlaying='y',
        side='right'
    ),
    legend=dict(x=0.05, y=0.95),
    margin=dict(l=40, r=40, t=60, b=40)
)
fig_l.show()


# 🔬 Phase 4: Behavioral Phenotypes & Cross-Segmentation (Roadmap M-P)
In this advanced section, we perform binning on key variables to uncover highly specific employee behavioral profiles. We cross-segment behaviors like caffeine reliance, extreme multitasking, and shift-based stress to see how the system recommendations adapt.


### 🔋 Plot M: Demographic Burnout (Sleep vs. Stress Bins)
* **Insight:** Does a "High Stress" state affect everyone equally? Here we segment stress levels into Low, Moderate, and High, crossed with Sleep Class to measure the ultimate toll on Cognitive Load.
* **Visualization:** Grouped Bar Chart mapping Binned Stress vs. Cognitive Load, categorized by Sleep Class.


In [15]:
df_m = df.copy()
df_m['Stress_Bin'] = pd.cut(df_m['Stress_Level_1_10'], bins=[0, 3, 6, 10], labels=['Low (1-3)', 'Moderate (4-6)', 'High (7-10)'])
grouped_m = df_m.groupby(['Stress_Bin', 'Sleep_Group'], observed=False)['Cognitive_Load_Score'].mean().reset_index()

fig_m = px.bar(
    grouped_m,
    x='Stress_Bin',
    y='Cognitive_Load_Score',
    color='Sleep_Group',
    barmode='group',
    color_discrete_sequence=['#e71d36', '#ff9f1c', '#2ec4b6'],
    labels={
        'Stress_Bin': 'Binned Stress Level',
        'Cognitive_Load_Score': 'Average Cognitive Load Score',
        'Sleep_Group': 'Sleep Class'
    },
    title='<b>Plot M: Demographic Burnout: How Sleep Buffers Binned Stress Levels</b>'
)
fig_m.update_layout(title_font_size=18, margin=dict(l=40, r=40, t=60, b=40))
fig_m.show()


### ☕ Plot N: The Caffeine Masking Effect (Awake Bins vs. Caffeine)
* **Insight:** By binning hours awake and caffeine intake, we see if heavy caffeine users actually maintain better accuracy during long shifts, or if the stimulant simply masks a catastrophic error rate.
* **Visualization:** Heatmap of Binned Hours Awake vs. Caffeine User Type, showing Average Error Rate.


In [16]:
df_n = df.copy()
df_n['Hours_Awake_Bin'] = pd.cut(df_n['Hours_Awake'], bins=[0, 8, 11, 14, 24], labels=['≤8h (Fresh)', '9-11h', '12-14h', '≥15h (Exhausted)'])
df_n['Caffeine_Bin'] = pd.cut(df_n['Caffeine_Intake_Cups'], bins=[-1, 0, 2, 10], labels=['None (0)', 'Light (1-2)', 'Heavy (3+)'])

pivot_n = df_n.groupby(['Caffeine_Bin', 'Hours_Awake_Bin'], observed=False)['Error_Rate'].mean().unstack().fillna(0)

fig_n = px.imshow(
    pivot_n * 100,
    labels=dict(x="Hours Awake Class", y="Caffeine User Type", color="Error Rate (%)"),
    title="<b>Plot N: Caffeine Masking Effect on Fatigue Error Rates</b>",
    color_continuous_scale="Reds",
    text_auto=".1f"
)
fig_n.update_layout(title_font_size=18, margin=dict(l=40, r=40, t=60, b=40))
fig_n.show()


### 🔀 Plot O: The "Overloaded Phenotype" (Decisions vs. Switching)
* **Insight:** We segment employees into volume bins (Decisions) and friction bins (Task Switches) to isolate the "High Volume + High Switching" phenotype. This is the profile most likely to receive a "Take Break" system action.
* **Visualization:** 2D Bubble Plot of Decision Volume vs. Switching Volume, sized by Fatigue Score and colored by System Action.


In [17]:
df_o = df.copy()
df_o['Decision_Bin'] = pd.qcut(df_o['Decisions_Made'], q=4, labels=['Low Vol', 'Med Vol', 'High Vol', 'Extreme Vol'])
df_o['Switch_Bin'] = pd.cut(df_o['Task_Switches'], bins=[-1, 10, 20, 50], labels=['Low Friction (<10)', 'Med Friction (10-20)', 'High Friction (>20)'])

# Group by the bins and find the most common system recommendation, plus average fatigue
grouped_o = df_o.groupby(['Decision_Bin', 'Switch_Bin'], observed=False).agg({
    'Decision_Fatigue_Score': 'mean',
    'System_Recommendation': lambda x: x.mode()[0] if not x.mode().empty else 'Continue'
}).reset_index()

fig_o = px.scatter(
    grouped_o,
    x='Switch_Bin',
    y='Decision_Bin',
    size='Decision_Fatigue_Score',
    color='System_Recommendation',
    color_discrete_map=PALETTE_RECOMMENDATION,
    size_max=30,
    labels={
        'Switch_Bin': 'Task Switching Friction (Bins)',
        'Decision_Bin': 'Decision Volume (Quartiles)',
        'Decision_Fatigue_Score': 'Avg Fatigue Score',
        'System_Recommendation': 'Majority System Action'
    },
    title='<b>Plot O: The Overloaded Phenotype: Mapping System Triggers to Workload Bins</b>',
    opacity=0.9
)
fig_o.update_traces(marker=dict(line=dict(width=0)))
fig_o.update_layout(title_font_size=18, margin=dict(l=40, r=40, t=60, b=40))
fig_o.show()


### 🌙 Plot P: Shift Disparities (Time of Day vs. Stress Bins)
* **Insight:** Are night-shift workers more stressed than morning workers under the same objective conditions? We plot Time of Day against our new Stress Bins to see the distribution of system recommendations.
* **Visualization:** 100% Stacked Bar chart showing the proportion of System Actions across Time of Day, faceted by Stress Level.


In [18]:
df_p = df.copy()
df_p['Stress_Bin'] = pd.cut(df_p['Stress_Level_1_10'], bins=[0, 3, 6, 10], labels=['Low (1-3)', 'Moderate (4-6)', 'High (7-10)'])

# Calculate proportions of System Recommendations
action_counts = df_p.groupby(['Stress_Bin', 'Time_of_Day', 'System_Recommendation'], observed=False).size().reset_index(name='count')
action_totals = df_p.groupby(['Stress_Bin', 'Time_of_Day'], observed=False).size().reset_index(name='total')
action_merged = pd.merge(action_counts, action_totals, on=['Stress_Bin', 'Time_of_Day'])
action_merged['Proportion'] = action_merged['count'] / action_merged['total']

fig_p = px.bar(
    action_merged,
    x='Time_of_Day',
    y='Proportion',
    color='System_Recommendation',
    facet_col='Stress_Bin',
    color_discrete_map=PALETTE_RECOMMENDATION,
    labels={
        'Time_of_Day': 'Shift (Time of Day)',
        'Proportion': 'Proportion of Alerts',
        'System_Recommendation': 'System Action'
    },
    title='<b>Plot P: Shift Disparities: System Alerts Across Shifts at Equal Stress Levels</b>'
)
fig_p.update_layout(title_font_size=18, yaxis=dict(tickformat='.0%'), margin=dict(l=40, r=40, t=60, b=40))
fig_p.show()
